In [1]:
import os
import datetime
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

In [2]:
###################################################################################################
## global variables
###################################################################################################
BASEPATH = "Path/To/Dataset"


INPUT_COLUMNS = [  # comment columns to train without them
    'left_back_roll', 'left_back_pitch', 'left_back_yaw',
    'left_mid_roll', 'left_mid_pitch', 'left_mid_yaw',  # new
    'left_front_roll', 'left_front_pitch', 'left_front_yaw',
    # 'left_imu_roll', 'left_imu_pitch', 'left_imu_yaw',
    'left_imu_accel_x', 'left_imu_accel_y', 'left_imu_accel_z',
    'left_imu_gyro_x', 'left_imu_gyro_y', 'left_imu_gyro_z',
    'right_back_roll', 'right_back_pitch', 'right_back_yaw',
    'right_mid_roll', 'right_mid_pitch', 'right_mid_yaw',  # new
    'right_front_roll', 'right_front_pitch', 'right_front_yaw',
    # 'right_imu_roll', 'right_imu_pitch', 'right_imu_yaw',
    'right_imu_accel_x', 'right_imu_accel_y', 'right_imu_accel_z',
    'right_imu_gyro_x', 'right_imu_gyro_y', 'right_imu_gyro_z',
    'left_x', 'left_y', 'left_z',
    'right_x', 'right_y', 'right_z',
    'z_difference'
]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda:0


In [7]:
###################################################################################################
## training and testing functions: create_dataset, train, test, extract_dataset_sample_names
###################################################################################################

def create_dataset(large_x, large_y, device):
    X = large_x.astype(np.float32)
    y = large_y.astype(np.float32)

    # Convert numpy arrays to PyTorch tensors
    X_tensor = torch.tensor(X).unsqueeze(1)  # Add a channel dimension
    y_tensor = torch.tensor(y).unsqueeze(1)  # Make y a column vector

    X_tensor = X_tensor.to(device)
    y_tensor = y_tensor.to(device)

    # Create a DataLoader
    dataset = TensorDataset(X_tensor, y_tensor)

    return dataset


def train(train_loader, model, optimizer, loss_fn, device):
    size = len(train_loader.dataset)
    model.train()
    running_loss = 0.0
    for batch, (X, y) in enumerate(train_loader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred_y = model(X)
        loss = loss_fn(pred_y, y)

        # backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    return running_loss


def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


def unparquet_trial_data(basepath, exercise, output_column):
    print(f"Loading training data with exercise={exercise} and output_column={output_column}...")
    target_folder = os.path.join(basepath, "parquet", f"training_{exercise}")
    all_inputs = []
    all_outputs = []
    for filename in os.listdir(target_folder):
        df = pd.read_parquet(os.path.join(target_folder, filename))
        input_df = df[INPUT_COLUMNS]
        output_df = df[output_column]
        all_inputs.append(input_df.to_numpy())
        all_outputs.append(output_df.to_numpy())
        
    print("samples", len(all_inputs))
    large_X = np.concatenate(all_inputs)  # all input angles and z_difference
    large_Y = np.concatenate(all_outputs)  # com_y
    return large_X, large_Y


def unparquet_validation_data(basepath, exercise, output_column):
    print(f"Loading test data with exercise={exercise} and output_column={output_column}...")
    target_folder = os.path.join(basepath, "parquet", f"test_{exercise}")
    Xs = {}  # all input angles and z_difference
    ys = {}  # com_y
    for filename in os.listdir(target_folder):
        df = pd.read_parquet(os.path.join(target_folder, filename))
        input_df = df[INPUT_COLUMNS]
        output_df = df[output_column]
        part_id = filename.split("_")[0]
        Xs[part_id] = input_df.to_numpy()
        ys[part_id] = output_df.to_numpy()

    return Xs, ys


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DeeperConvLSTMNet(nn.Module):
    #def __init__(self, input_channels=1, lstm_hidden_size=128, lstm_num_layers=2):
    def __init__(self, input_channels=1, lstm_hidden_size=256, lstm_num_layers=4):
        """
        A deeper hybrid CNN-LSTM model for sequence data.

        Args:
            input_channels (int): Number of channels in the input sequence.
            lstm_hidden_size (int): The number of features in the LSTM hidden state.
            lstm_num_layers (int): Number of recurrent LSTM layers.
        """
        super(DeeperConvLSTMNet, self).__init__()

        # --- Deeper CNN Feature Extractor ---
        # Each block consists of two conv layers and one pooling layer.

        # Block 1: (batch, 1, seq_len) -> (batch, 32, seq_len/2)
        self.conv_unit_1 = nn.Sequential(
            nn.Conv1d(in_channels=input_channels, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        )

        # Block 2: (batch, 32, seq_len/2) -> (batch, 64, seq_len/4)
        self.conv_unit_2 = nn.Sequential(
            nn.Conv1d(in_channels=32, out_channels=48, kernel_size=3, padding=1),
            nn.BatchNorm1d(48),
            nn.ReLU(),
            nn.Conv1d(in_channels=48, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        )

        # Block 3: (batch, 64, seq_len/4) -> (batch, 96, seq_len/8)
        self.conv_unit_3 = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=80, kernel_size=3, padding=1),
            nn.BatchNorm1d(80),
            nn.ReLU(),
            nn.Conv1d(in_channels=80, out_channels=96, kernel_size=3, padding=1),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        )

        # Block 4: (batch, 96, seq_len/8) -> (batch, 128, seq_len/16)
        self.conv_unit_4 = nn.Sequential(
            nn.Conv1d(in_channels=96, out_channels=112, kernel_size=3, padding=1),
            nn.BatchNorm1d(112),
            nn.ReLU(),
            nn.Conv1d(in_channels=112, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        )

        """
        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # Block 2: (batch, 32, seq_len/2) -> (batch, 64, seq_len/4)
        self.conv3 = nn.Conv1d(in_channels=32, out_channels=48, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(in_channels=48, out_channels=64, kernel_size=3, padding=1)

        # Block 3: (batch, 64, seq_len/4) -> (batch, 96, seq_len/8)
        self.conv5 = nn.Conv1d(in_channels=64, out_channels=80, kernel_size=3, padding=1)
        self.conv6 = nn.Conv1d(in_channels=80, out_channels=96, kernel_size=3, padding=1)
        
        # Block 4: (batch, 96, seq_len/8) -> (batch, 128, seq_len/16)
        self.conv7 = nn.Conv1d(in_channels=96, out_channels=112, kernel_size=3, padding=1)
        self.conv8 = nn.Conv1d(in_channels=112, out_channels=128, kernel_size=3, padding=1)

        # A single pooling layer definition, reused after each block
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        """

        # LSTM Sequence Processor
        self.lstm = nn.LSTM(
            input_size=128,  # Must match the out_channels of the last conv layer (conv8)
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=0.3 if lstm_num_layers > 1 else 0
        )

        # Fully-Connected Regression Head
        self.fc1 = nn.Linear(lstm_hidden_size, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        # x starts as (batch_size, input_channels, sequence_length)

        # 1. Pass through CNN feature extractor
        # Block 1

        """
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        # Block 2
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)
        
        # Block 3
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x = self.pool(x)
        
        # Block 4
        x = F.relu(self.conv7(x))
        x = F.relu(self.conv8(x))
        x = self.pool(x)
        """

        x = self.conv_unit_1(x)
        x = self.conv_unit_2(x)
        x = self.conv_unit_3(x)
        x = self.conv_unit_4(x)

        # After 4 pooling layers, shape is (batch, 128, sequence_length / 16)

        # 2. Reshape for LSTM
        # Current shape: (batch, channels, seq) -> LSTM needs: (batch, seq, channels/features)
        x = x.permute(0, 2, 1)

        # 3. Pass through LSTM
        lstm_out, (h_n, c_n) = self.lstm(x)

        # 4. Use the output from the last time step
        last_lstm_output = lstm_out[:, -1, :]

        # 5. Pass through the final fully-connected layers
        x = self.dropout(F.relu(self.fc1(last_lstm_output)))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x



def train_nn(dataset, model_name, device, training_epochs=15, batch_size=2048):
    np.random.seed(42)
    torch.manual_seed(42)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = DeeperConvLSTMNet().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    #scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)
    loss_fn = nn.HuberLoss()  # Robust to outliers vs MSE

    num_epochs = training_epochs
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}\n-------------------------------")
        running_loss = train(train_loader, model, optimizer, loss_fn, device)
        test(val_loader, model, loss_fn)
        avg_loss = running_loss / len(train_loader)
        print(f"[{datetime.datetime.now()}] Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}")

    print(f"[{datetime.datetime.now()}] Training finished!")

    torch.save(model.state_dict(), f"{model_name}.model")
    return model




In [9]:
def validate(dataloader, model):
    model.eval()
    predicteds = []
    expecteds = []

    with torch.no_grad():
        for X, y in dataloader:
            print(X.shape)
            X, y = X.to(device), y.to(device)
            pred = model(X).to("cpu")
            y = y.to("cpu")
            pred = pred.detach().numpy()

            predicteds.append(pred.squeeze(1))
            expecteds.append(y.numpy())

    return predicteds, expecteds

In [10]:
import numpy as np
from scipy.signal import butter, sosfiltfilt


def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    sos = butter(order, normal_cutoff, btype='low', analog=False, output='sos')
    y = sosfiltfilt(sos, data)
    return y

In [ ]:
# Stage 1: train the DNN model

def train_all_nn_models():
    output_names = [
        'com_x',
        'com_y',
        'com_z'
    ]

    exercises = [
        'Sta',
        'Ste',
        'W'
    ]
    models = {}
    for output_name in output_names:
        for exercise in exercises:
            model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_20_lstm4'
            print(f"Training model {model_name}...")

            large_X, large_Y = unparquet_trial_data(BASEPATH, exercise, output_name)
            scaler = StandardScaler()

            X = scaler.fit_transform(large_X)
            y = large_Y - np.mean(large_Y)

            dataset = create_dataset(X, y, device)
            model = train_nn(dataset, model_name, device, training_epochs=20, batch_size=2048)
            models[model_name] = model

    return models


train_all_nn_models()

In [14]:
# Stage 2: validate the DNN model


output_names = [
    'com_x',
    'com_y',
    'com_z'
]

exercises = [
    'Sta',
    'Ste',
    'W'
]

import numpy as np
from scipy.signal import butter, sosfiltfilt

scaler = StandardScaler()
for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        print(f"Testing model {model_name}...")

        model = DeeperConvLSTMNet().to(device)
        model.load_state_dict(
            torch.load(
                f"models/{model_name}.model",
                weights_only=True
            )
        )
        model.eval()

        # run model validation
        large_eval_Xs, large_eval_Ys = unparquet_validation_data(BASEPATH, exercise, output_name)

        expected_ys = []
        predicted_ys = []
        f_expected_ys = []
        f_predicted_ys = []
        part_ids = []

        for part_id in large_eval_Xs.keys():
            large_eval_X = large_eval_Xs[part_id]
            large_eval_Y = large_eval_Ys[part_id]

            X_eval = scaler.fit_transform(large_eval_X)
            y_eval = large_eval_Y - np.mean(large_eval_Y)

            dataset_eval = create_dataset(X_eval, y_eval, device)
            dataset_loader = DataLoader(dataset_eval, batch_size=2048, shuffle=False)

            predicteds, expecteds = validate(dataset_loader, model)

            ps = [float(p) for p in np.concatenate(predicteds)]
            es = [float(e[0]) for e in np.concatenate(expecteds)]

            predicted_ys.append(ps)
            expected_ys.append(es)
            part_ids.append(part_id)

            #print(test_status_df)

            cutoff_frequency = 25
            sampling_frequency = 500
            filter_order = 4

            filtered_predicted = butter_lowpass_filter(np.array(ps), cutoff_frequency, sampling_frequency,
                                                       order=filter_order)
            filtered_expected = butter_lowpass_filter(np.array(es), cutoff_frequency, sampling_frequency,
                                                      order=filter_order)

            filtered_predicted = [float(p) for p in filtered_predicted]
            filtered_expected = [float(e) for e in filtered_expected]

            f_predicted_ys.append(filtered_predicted)
            f_expected_ys.append(filtered_expected)

        model_test_result_df = pd.DataFrame(data={
            "id": part_ids,
            "expected": expected_ys,
            "predicted": predicted_ys,
            "filtered_expected": f_expected_ys,
            "filtered_predicted": f_predicted_ys
        })

        model_test_result_df.to_csv(f"results/test_result_{model_name}.csv")

Testing model Sta_Sta_com_x_centred_DeeperConvLSTMNet_e_10...


RuntimeError: Error(s) in loading state_dict for DeeperConvLSTMNet:
	Missing key(s) in state_dict: "lstm.weight_ih_l2", "lstm.weight_hh_l2", "lstm.bias_ih_l2", "lstm.bias_hh_l2", "lstm.weight_ih_l3", "lstm.weight_hh_l3", "lstm.bias_ih_l3", "lstm.bias_hh_l3". 
	size mismatch for lstm.weight_ih_l0: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in current model is torch.Size([1024, 128]).
	size mismatch for lstm.weight_hh_l0: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in current model is torch.Size([1024, 256]).
	size mismatch for lstm.bias_ih_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for lstm.bias_hh_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for lstm.weight_ih_l1: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in curren

In [ ]:
from ast import literal_eval
import pandas as pd
import numpy as np

output_names = [
    'com_x',
    'com_y',
    'com_z'
]

exercises = [
    'Sta',
    'Ste',
    'W'
]

best_identifiers = []


def calculate_statistics_for_result(filepath):
    result_df = pd.read_csv(filepath)
    d_mean = []
    d_std = []
    d_min = []
    d25 = []
    d50 = []
    d75 = []
    d_max = []

    e_mean = []
    e_std = []
    e_min = []
    e25 = []
    e50 = []
    e75 = []
    e_max = []

    p_mean = []
    p_std = []
    p_min = []
    p25 = []
    p50 = []
    p75 = []
    p_max = []

    #
    fd_mean = []
    fd_std = []
    fd_min = []
    fd25 = []
    fd50 = []
    fd75 = []
    fd_max = []

    fe_mean = []
    fe_std = []
    fe_min = []
    fe25 = []
    fe50 = []
    fe75 = []
    fe_max = []

    fp_mean = []
    fp_std = []
    fp_min = []
    fp25 = []
    fp50 = []
    fp75 = []
    fp_max = []

    best_identifier = None
    ids = []
    current_diffs = []
    minimum_val = 100_000
    identifier = result_df.id
    for k in range(len(result_df.predicted)):
        predicted = np.array(literal_eval(result_df.predicted.to_numpy()[k]))
        expected = np.array(literal_eval(result_df.expected.to_numpy()[k]))
        
  

        fpredicted = np.array(literal_eval(result_df.filtered_predicted.to_numpy()[k]))
        fexpected = np.array(literal_eval(result_df.filtered_expected.to_numpy()[k]))

        delta = np.abs(predicted - expected)  # center around expected
        fdelta = fpredicted - fexpected  # center around expected

        if np.abs(np.mean(fdelta)) < minimum_val:
            minimum_val = np.abs(np.mean(fdelta))
            best_identifier = identifier[k]

        current_diffs.append([fdelta, fexpected, fpredicted])

        df_row = pd.DataFrame(data={
            "p": predicted,
            'e': expected,
            'd': delta,
            "fp": fpredicted,
            'fe': fexpected,
            'fd': fdelta

        })
        described_stats = df_row.describe()

        d_mean.append(float(described_stats.d["mean"]))
        d_std.append(float(described_stats.d["std"]))
        d_min.append(float(described_stats.d["min"]))
        d25.append(float(described_stats.d["25%"]))
        d50.append(float(described_stats.d["50%"]))
        d75.append(float(described_stats.d["75%"]))
        d_max.append(float(described_stats.d["max"]))

        e_mean.append(float(described_stats.e["mean"]))
        e_std.append(float(described_stats.e["std"]))
        e_min.append(float(described_stats.e["min"]))
        e25.append(float(described_stats.e["25%"]))
        e50.append(float(described_stats.e["50%"]))
        e75.append(float(described_stats.e["75%"]))
        e_max.append(float(described_stats.e["max"]))

        p_mean.append(float(described_stats.p["mean"]))
        p_std.append(float(described_stats.p["std"]))
        p_min.append(float(described_stats.p["min"]))
        p25.append(float(described_stats.p["25%"]))
        p50.append(float(described_stats.p["50%"]))
        p75.append(float(described_stats.p["75%"]))
        p_max.append(float(described_stats.p["max"]))

        # filtered params
        fd_mean.append(float(described_stats.fd["mean"]))
        fd_std.append(float(described_stats.fd["std"]))
        fd_min.append(float(described_stats.fd["min"]))
        fd25.append(float(described_stats.fd["25%"]))
        fd50.append(float(described_stats.fd["50%"]))
        fd75.append(float(described_stats.fd["75%"]))
        fd_max.append(float(described_stats.fd["max"]))

        fe_mean.append(float(described_stats.fe["mean"]))
        fe_std.append(float(described_stats.fe["std"]))
        fe_min.append(float(described_stats.fe["min"]))
        fe25.append(float(described_stats.fe["25%"]))
        fe50.append(float(described_stats.fe["50%"]))
        fe75.append(float(described_stats.fe["75%"]))
        fe_max.append(float(described_stats.fe["max"]))

        fp_mean.append(float(described_stats.fp["mean"]))
        fp_std.append(float(described_stats.fp["std"]))
        fp_min.append(float(described_stats.fp["min"]))
        fp25.append(float(described_stats.fp["25%"]))
        fp50.append(float(described_stats.fp["50%"]))
        fp75.append(float(described_stats.fp["75%"]))
        fp_max.append(float(described_stats.fp["max"]))

        ids.append(str(result_df.id[k]))
    
    
    print("M = ", np.mean(expected))
    print("std = ", np.std(expected))
    
    best_identifiers.append(best_identifier)
    statistics_df = pd.DataFrame(
        data={
            "id": ids,
            "d_mean": d_mean,
            "d_std": d_std,
            "d_min": d_min,
            "d_q25": d25,
            "d_q50": d50,
            "d_q75": d75,
            "d_max": d_max,
            "e_mean": e_mean,
            "e_std": e_std,
            "e_min": e_min,
            "e_q25": e25,
            "e_q50": e50,
            "e_q75": e75,
            "e_max": e_max,
            "p_mean": p_mean,
            "p_std": p_std,
            "p_min": p_min,
            "p_q25": p25,
            "p_q50": p50,
            "p_q75": p75,
            "p_max": p_max,
            "fd_mean": fd_mean,
            "fd_std": fd_std,
            "fd_min": fd_min,
            "fd_q25": fd25,
            "fd_q50": fd50,
            "fd_q75": fd75,
            "fd_max": fd_max,
            "fe_mean": fe_mean,
            "fe_std": fe_std,
            "fe_min": fe_min,
            "fe_q25": fe25,
            "fe_q50": fe50,
            "fe_q75": fe75,
            "fe_max": fe_max,
            "fp_mean": fp_mean,
            "fp_std": fp_std,
            "fp_min": fp_min,
            "fp_q25": fp25,
            "fp_q50": fp50,
            "fp_q75": fp75,
            "fp_max": fp_max
        }
    )

    return statistics_df, current_diffs


all_diffs = {}
for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        print(f"Calculating statistics for results {model_name}...")
        filepath = f"results/test_result_{model_name}.csv"
        statistics_df, current_diffs = calculate_statistics_for_result(filepath)
        all_diffs[f"{output_name}-{exercise}"] = current_diffs
        statistics_df.to_csv(f"results/statistics_{model_name}.csv")

In [16]:
lefts = [a[0] for a in all_diffs["com_x-W"]]
rights = [a[1] for a in all_diffs["com_x-W"]]

difference_values = np.concatenate(lefts)
expected_values = np.concatenate(rights)



In [ ]:
idx = 0
on_average_candidates = []
for output_name in output_names:
    for exercise in exercises:
        on_average_candidates.append([output_name, exercise, best_identifiers[idx]])
        idx += 1

on_average_candidates

In [ ]:
import numpy as np
from collections import defaultdict
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from scipy import stats

def get_histogram_indices(arr, bins, difference_values=None):
    bin_indices = np.digitize(arr, bins)

    indices_by_bin = defaultdict(list)
    for idx, bin_idx in enumerate(bin_indices):
        indices_by_bin[bins[bin_idx - 1]].append(difference_values[idx])

    return dict(indices_by_bin)


half_len = 400
bin_range_min = -half_len
bin_range_max = half_len
bin_counts = int((bin_range_max - bin_range_min) * 0.2 + 1)

bins = np.linspace(bin_range_min, bin_range_max, bin_counts)  # 10 bins
print(len(bins))

propper_naming_map = {
    "com_x-Sta": "Single Leg Stance CoM X-Axis",
    "com_y-Sta": "Single Leg Stance CoM Y-Axis",
    "com_z-Sta": "Single Leg Stance CoM Z-Axis",
    "com_x-Ste": "Stepping CoM X-Axis",
    "com_y-Ste": "Stepping CoM Y-Axis",
    "com_z-Ste": "Stepping CoM Z-Axis",
    "com_x-W": "Waves CoM X-Axis",
    "com_y-W": "Waves CoM Y-Axis",
    "com_z-W": "Waves CoM Z-Axis",

}


def mask_of_n_sigma(arr, n=1):
    """
    Select samples within n standard deviation of the mean (n=1 by default). 
    Returns: mask of samples within [μ - 0.5σ, μ + 0.5σ]
    """
    if n == 0:
        mask = np.abs(arr - np.mean(arr)) == np.abs(arr - np.mean(arr))
        return mask, 1, np.std(arr)
    
    nsigma = n * np.std(arr)
    print("s", nsigma)
    mask = np.abs(arr - np.mean(arr)) <= 0.5 * nsigma
    
    
    return mask, nsigma, np.std(arr)


all_statistic_errors = {}
add_statistic_measures = {}
all_bin_stats = {}
for output_name in output_names:
    for exercise in exercises:
        target_dataset_name = f"{output_name}-{exercise}"

        lefts = [a[0] for a in all_diffs[target_dataset_name]]
        rights = [a[1] for a in all_diffs[target_dataset_name]]
        
        
        difference_values = np.concatenate(lefts)
        expected_values = np.concatenate(rights)
        grouped_indices = get_histogram_indices(expected_values, bins, difference_values)

        expected_std = np.std(expected_values)
        
        statistics_errs = {}
        add_statistics_errs = {}
        sorted_group = {k: v for k, v in sorted(grouped_indices.items(), key=lambda item: item[0])}

        bin_counts = {k: len(v) for k, v in sorted_group.items()}
        all_bin_stats[target_dataset_name] = bin_counts
        for pos_value, errors in sorted_group.items():
            q25 = np.quantile(errors, q=0.25)
            q75 = np.quantile(errors, q=0.75)
            q50 = np.median(errors)
            q02 = np.quantile(errors, q=0.02)
            q98 = np.quantile(errors, q=0.98)
            statistics_errs[pos_value] = [q02, q25, q50, q75, q98]

        _errors2 = np.array(difference_values) + expected_values
        

        n_sigma_mask, sigma_value, std = mask_of_n_sigma(_errors2)
        rms = root_mean_squared_error(_errors2[n_sigma_mask], expected_values[n_sigma_mask])
        
        _errors = _errors2[n_sigma_mask] - expected_values[n_sigma_mask]

        var = np.var(_errors)
        se = np.sqrt(var / len(_errors))
        mean = np.mean(_errors)
        rmse = np.sqrt(np.sum(_errors ** 2.0) / len(_errors))

        expected_std = np.std(expected_values[n_sigma_mask])
        
        mm, nsigma, std = mask_of_n_sigma(expected_values)
        expected_std = np.std(expected_values)
        expected_mean = np.mean(expected_values)
        add_statistic_measures[target_dataset_name] = [mean, var, rms, se, expected_std, sigma_value, expected_mean, r2]

        all_statistic_errors[target_dataset_name] = statistics_errs

print("done")

In [ ]:
from ast import literal_eval
import pandas as pd
import numpy as np

output_names = [
    'com_x',
    'com_y',
    'com_z'
]

exercises = [
    'Sta',
    'Ste',
    'W'
]

all_diffs = []


def calculate_statistics_for_result(filepath):
    result_df = pd.read_csv(filepath)
    d_mean = []
    d_std = []
    d_min = []
    d25 = []
    d50 = []
    d75 = []
    d_max = []

    e_mean = []
    e_std = []
    e_min = []
    e25 = []
    e50 = []
    e75 = []
    e_max = []

    p_mean = []
    p_std = []
    p_min = []
    p25 = []
    p50 = []
    p75 = []
    p_max = []

    #
    fd_mean = []
    fd_std = []
    fd_min = []
    fd25 = []
    fd50 = []
    fd75 = []
    fd_max = []

    fe_mean = []
    fe_std = []
    fe_min = []
    fe25 = []
    fe50 = []
    fe75 = []
    fe_max = []

    fp_mean = []
    fp_std = []
    fp_min = []
    fp25 = []
    fp50 = []
    fp75 = []
    fp_max = []

    ids = []
    for k in range(len(result_df.predicted)):
        predicted = np.array(literal_eval(result_df.predicted.to_numpy()[k]))
        expected = np.array(literal_eval(result_df.expected.to_numpy()[k]))

        fpredicted = np.array(literal_eval(result_df.filtered_predicted.to_numpy()[k]))
        fexpected = np.array(literal_eval(result_df.filtered_expected.to_numpy()[k]))

        delta = np.abs(predicted - expected)  # center around expected
        fdelta = fpredicted - fexpected  # center around expected

        all_diffs.append([fdelta, expected])

        df_row = pd.DataFrame(data={
            "p": predicted,
            'e': expected,
            'd': delta,
            "fp": fpredicted,
            'fe': fexpected,
            'fd': fdelta

        })
        described_stats = df_row.describe()

        d_mean.append(float(described_stats.d["mean"]))
        d_std.append(float(described_stats.d["std"]))
        d_min.append(float(described_stats.d["min"]))
        d25.append(float(described_stats.d["25%"]))
        d50.append(float(described_stats.d["50%"]))
        d75.append(float(described_stats.d["75%"]))
        d_max.append(float(described_stats.d["max"]))

        e_mean.append(float(described_stats.e["mean"]))
        e_std.append(float(described_stats.e["std"]))
        e_min.append(float(described_stats.e["min"]))
        e25.append(float(described_stats.e["25%"]))
        e50.append(float(described_stats.e["50%"]))
        e75.append(float(described_stats.e["75%"]))
        e_max.append(float(described_stats.e["max"]))

        p_mean.append(float(described_stats.p["mean"]))
        p_std.append(float(described_stats.p["std"]))
        p_min.append(float(described_stats.p["min"]))
        p25.append(float(described_stats.p["25%"]))
        p50.append(float(described_stats.p["50%"]))
        p75.append(float(described_stats.p["75%"]))
        p_max.append(float(described_stats.p["max"]))

        # filtered params
        fd_mean.append(float(described_stats.fd["mean"]))
        fd_std.append(float(described_stats.fd["std"]))
        fd_min.append(float(described_stats.fd["min"]))
        fd25.append(float(described_stats.fd["25%"]))
        fd50.append(float(described_stats.fd["50%"]))
        fd75.append(float(described_stats.fd["75%"]))
        fd_max.append(float(described_stats.fd["max"]))

        fe_mean.append(float(described_stats.fe["mean"]))
        fe_std.append(float(described_stats.fe["std"]))
        fe_min.append(float(described_stats.fe["min"]))
        fe25.append(float(described_stats.fe["25%"]))
        fe50.append(float(described_stats.fe["50%"]))
        fe75.append(float(described_stats.fe["75%"]))
        fe_max.append(float(described_stats.fe["max"]))

        fp_mean.append(float(described_stats.fp["mean"]))
        fp_std.append(float(described_stats.fp["std"]))
        fp_min.append(float(described_stats.fp["min"]))
        fp25.append(float(described_stats.fp["25%"]))
        fp50.append(float(described_stats.fp["50%"]))
        fp75.append(float(described_stats.fp["75%"]))
        fp_max.append(float(described_stats.fp["max"]))

        ids.append(str(result_df.id[k]))

    statistics_df = pd.DataFrame(
        data={
            "id": ids,
            "d_mean": d_mean,
            "d_std": d_std,
            "d_min": d_min,
            "d_q25": d25,
            "d_q50": d50,
            "d_q75": d75,
            "d_max": d_max,
            "e_mean": e_mean,
            "e_std": e_std,
            "e_min": e_min,
            "e_q25": e25,
            "e_q50": e50,
            "e_q75": e75,
            "e_max": e_max,
            "p_mean": p_mean,
            "p_std": p_std,
            "p_min": p_min,
            "p_q25": p25,
            "p_q50": p50,
            "p_q75": p75,
            "p_max": p_max,
            "fd_mean": fd_mean,
            "fd_std": fd_std,
            "fd_min": fd_min,
            "fd_q25": fd25,
            "fd_q50": fd50,
            "fd_q75": fd75,
            "fd_max": fd_max,
            "fe_mean": fe_mean,
            "fe_std": fe_std,
            "fe_min": fe_min,
            "fe_q25": fe25,
            "fe_q50": fe50,
            "fe_q75": fe75,
            "fe_max": fe_max,
            "fp_mean": fp_mean,
            "fp_std": fp_std,
            "fp_min": fp_min,
            "fp_q25": fp25,
            "fp_q50": fp50,
            "fp_q75": fp75,
            "fp_max": fp_max
        }
    )

    return statistics_df


for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        print(f"Calculating statistics for results {model_name}...")
        filepath = f"results/test_result_simple_model_{model_name}.csv"
        statistics_df = calculate_statistics_for_result(filepath)
        statistics_df.to_csv(f"results/statistics_simple_model_{model_name}.csv")

In [ ]:
for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        print(f"Calculating statistics for results {model_name}...")
        filepath = f"results/test_result_simple_model_{model_name}.csv"
        result_df = pd.read_csv(filepath)

In [ ]:
for key, value in add_statistic_measures.items():
    print(key, ":")
    print(f" M: {value[0]}")
    print(f" SD: {np.sqrt(value[1])}")
    print(f" RMSE: {value[2]}")
    print(f" SE: {value[3]}")
    print(f" eSD: {value[4]}")
    print(f" v_sigma: {value[5]}")
    print(f" v_expected: {value[6]}")
    print(f"  v_range: {value[6]} pm {value[5]*0.5}")
    print()


In [11]:
# IMPORTANT

import json

with open("results/prediction_error_bin_distributions.json", "w") as f:
    json.dump(all_bin_stats, f)

In [12]:
# IMPORTANT

import json

normalized_all_statistic_errors = {}

for key, bin_data in all_statistic_errors.items():
    n_bin_data = {}
    for bin, statistics in bin_data.items():
        n_bin_data[int(bin)] = statistics

    normalized_all_statistic_errors[key] = n_bin_data

with open("results/prediction_error_statistics.json", "w") as f:
    json.dump(normalized_all_statistic_errors, f)

In [ ]:
for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        filepath = f"results/statistics_{model_name}.csv"
        statistics_df = pd.read_csv(filepath)
        print(model_name)
        print(statistics_df[["d_mean", "fd_mean"]].describe()["d_mean"][["mean", "std"]])
        print(np.mean(np.abs((statistics_df.d_mean))))

        print()

In [ ]:
## Train and validate simple models


import pandas as pd
from sklearn import linear_model

output_names = [
    'com_x',
    'com_y',
    'com_z'
]

exercises = [
    'Sta',
    'Ste',
    'W'
]
scaler = StandardScaler()

simple_models = {}
for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        print(f"Testing model {model_name}...")

        # run model validation
        large_X, large_Y = unparquet_trial_data(BASEPATH, exercise, output_name)
        scaler = StandardScaler()
        X = large_X  # scaler.fit_transform(large_X)
        y = large_Y - np.mean(large_Y)
        simple_model = linear_model.LinearRegression()
        simple_model.fit(X, y)
        simple_models[f"{output_name}_{exercise}"] = simple_model

        large_eval_Xs, large_eval_Ys = unparquet_validation_data(BASEPATH, exercise, output_name)

        expected_ys = []
        predicted_ys = []
        f_expected_ys = []
        f_predicted_ys = []
        part_ids = []

        for part_id in large_eval_Xs.keys():
            large_eval_X = large_eval_Xs[part_id]
            large_eval_Y = large_eval_Ys[part_id]

            X_eval = large_eval_X  #scaler.fit_transform(large_eval_X) 
            y_eval = large_eval_Y - np.mean(large_eval_Y)
            y_pred = simple_model.predict(X_eval)

            predicteds, expecteds = y_pred, y_eval

            ps = [float(p) for p in predicteds]
            es = [float(e) for e in expecteds]

            predicted_ys.append(ps)
            expected_ys.append(es)
            part_ids.append(part_id)

            cutoff_frequency = 25
            sampling_frequency = 500
            filter_order = 4

            filtered_predicted = butter_lowpass_filter(np.array(ps), cutoff_frequency, sampling_frequency,
                                                       order=filter_order)
            filtered_expected = butter_lowpass_filter(np.array(es), cutoff_frequency, sampling_frequency,
                                                      order=filter_order)

            filtered_predicted = [float(p) for p in filtered_predicted]
            filtered_expected = [float(e) for e in filtered_expected]

            f_predicted_ys.append(filtered_predicted)
            f_expected_ys.append(filtered_expected)

        model_test_result_df = pd.DataFrame(data={
            "id": part_ids,
            "expected": expected_ys,
            "predicted": predicted_ys,
            "filtered_expected": f_expected_ys,
            "filtered_predicted": f_predicted_ys
        })

        model_test_result_df.to_csv(f"results/test_result_simple_model_{model_name}.csv")



In [ ]:
for output_name in output_names:
    for exercise in exercises:
        model_name = exercise + F'_{exercise}_{output_name}_centred_DeeperConvLSTMNet_e_10'
        filepath = f"results/statistics_simple_model_{model_name}.csv"
        statistics_df = pd.read_csv(filepath)

        filepath = f"results/statistics_{model_name}.csv"
        statistics_ai_df = pd.read_csv(filepath)

        final_name = "_".join(model_name.split("_")[1:])
        print(final_name)
        print("Simple")
        print(statistics_df[["d_mean", "fd_mean"]].describe()["d_mean"][["mean", "std"]])

        print("AI")
        print(statistics_ai_df[["d_mean", "fd_mean"]].describe()["d_mean"][["mean", "std"]])

        print()